# Nepali Speech to Text (नेपाली स्पीच टु टेक्स्ट)

Try [BlackfireAI/nepali-asr-xlsr-300m](https://huggingface.co/BlackfireAI/nepali-asr-xlsr-300m), an open Nepali speech recognition model by [Blackfire A.I.](https://blackfire.com.np)

**How to use:** click **Runtime → Run all**, then upload a Nepali audio file (wav, mp3, m4a) when asked. No GPU needed.

**Nepali only.** It does not handle English words inside Nepali speech. No punctuation; numbers come out as words.

Free for research and non-commercial use (CC BY-NC-SA 4.0). For commercial use, contact hello@blackfire.com.np. Please do not upload private or sensitive recordings.

In [ ]:
# 1. Install the tested transformers version (takes about a minute)
!pip install -q transformers==5.17.0

In [ ]:
# 2. Load the model (downloads about 1.2 GB the first time)
import torch
from transformers import AutoModelForCTC, AutoProcessor

name = "BlackfireAI/nepali-asr-xlsr-300m"
processor = AutoProcessor.from_pretrained(name)
model = AutoModelForCTC.from_pretrained(name).eval()
print("Model ready")

In [ ]:
# 3. Upload a Nepali recording and transcribe it
# Works with wav, mp3, m4a, ogg, and WhatsApp voice notes (.mp4 / .opus)
import subprocess, warnings
import soundfile as sf
from google.colab import files
warnings.filterwarnings("ignore")

def load_16k_mono(path):
    wav = path + ".16k.wav"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path,
                    "-ac", "1", "-ar", "16000", wav], check=True)
    audio, _ = sf.read(wav, dtype="float32")
    return audio

def transcribe(path, chunk_seconds=30):
    audio = load_16k_mono(path)
    print(f"Audio length: {len(audio) / 16000:.1f} seconds")
    step = chunk_seconds * 16000
    parts = []
    for start in range(0, len(audio), step):
        chunk = audio[start:start + step]
        if len(chunk) < 1600:
            continue
        inputs = processor(chunk, sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            logits = model(inputs.input_values).logits
        parts.append(processor.batch_decode(logits.argmax(-1).numpy())[0].strip())
    text = " ".join(p for p in parts if p)
    return text or "(no speech recognised)"

uploaded = files.upload()
for filename in uploaded:
    print(f"\n{filename}")
    print(transcribe(filename))

Found a mistake? Please open an issue with the audio (if you can share it) and the wrong output: [github.com/BlackfireAI/nepali-asr-xlsr/issues](https://github.com/BlackfireAI/nepali-asr-xlsr/issues)